In [0]:
%pip install statsmodels pmdarima

In [0]:
import mlflow, pickle
import pandas as pd
import numpy as np
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")

# Cell 1: Find the best run by RMSE
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"]
)
best_run = runs[0]
best_run_id = best_run.info.run_id
print(f"Best run: {best_run.data.tags.get('mlflow.runName')}  RMSE: {best_run.data.metrics['rmse']:.4f}")



## 📝 Line-by-Line Explanation of Cell 2: Finding Best Model

### **Lines 1-3: Import Required Libraries**
```python
import mlflow, pickle
import pandas as pd
import numpy as np
from mlflow.tracking import MlflowClient
```
**What each import does:**
- **`mlflow`**: MLflow main library for experiment tracking
- **`pickle`**: Python serialization library to load saved models
- **`pandas as pd`**: Data manipulation library for DataFrames
- **`numpy as np`**: Numerical computing library (used for calculations)
- **`MlflowClient`**: MLflow's client API to query experiments, runs, and artifacts

---

### **Line 6: Create MLflow Client**
```python
client = MlflowClient()
```
**What it does:** Creates a client object to interact with MLflow tracking server.
- This client lets you programmatically search runs, download artifacts, and retrieve metrics
- It connects to the Databricks MLflow tracking server automatically

---

### **Line 7: Get Experiment by Name**
```python
experiment = client.get_experiment_by_name("/Users/santhoshnagendrarajan@gmail.com/weather-sarima")
```
**What it does:** Retrieves the experiment object by its name.
- **Experiment name:** `/Users/santhoshnagendrarajan@gmail.com/weather-sarima`
- This experiment was created in the SARIMA Training notebook (Cell 7)
- The `experiment` object contains:
  - `experiment_id`: Unique identifier
  - Metadata like creation time, tags, etc.

**Why this matters:** You need the experiment ID to search for runs within that experiment.

---

### **Lines 10-13: Search for Best Run**
```python
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"]
)
```
**What it does:** Queries all runs in the experiment and sorts by RMSE.

**Line-by-line breakdown:**
- **Line 10:** Call the `search_runs()` method
- **Line 11:** Filter to only runs from this specific experiment
  - `experiment_ids=[...]`: Takes a list of experiment IDs
  - `experiment.experiment_id`: Gets the ID from the experiment object (e.g., "123456789")
- **Line 12:** Sort results by RMSE in ascending order
  - `"metrics.rmse ASC"`: Sort by the RMSE metric, lowest first
  - **ASC** = ascending (lowest RMSE = best model)
  - Could also use `"DESC"` for descending

**Result:** Returns a list of `Run` objects sorted by performance, with the best model first.

---

### **Line 14: Get Best Run**
```python
best_run = runs[0]
```
**What it does:** Selects the first run from the sorted list.
- Since we sorted by `"rmse ASC"`, the first run has the **lowest RMSE**
- `best_run` is a `Run` object containing all run data

**Run object structure:**
```
best_run
├── info          (run metadata)
│   ├── run_id    (unique identifier)
│   ├── start_time
│   └── status
├── data          (logged content)
│   ├── metrics   (RMSE, MAE, MAPE)
│   ├── params    (p, d, q, P, S, AIC)
│   └── tags      (run name, user)
└── artifacts     (forecast plots, model files)
```

---

### **Line 15: Extract Run ID**
```python
best_run_id = best_run.info.run_id
```
**What it does:** Extracts the unique run ID for later use.
- **`best_run.info`**: Accesses metadata about the run
- **`.run_id`**: Gets the unique identifier (UUID format)
- Example: `"57024b16e6c54c61a5d9e45d9f1f5cc8"`

**Why save this?** You need the run ID to:
1. Download artifacts (model file, plots)
2. Track which model produced each forecast
3. Create lineage between predictions and training runs

---

### **Line 16: Print Best Run Info**
```python
print(f"Best run: {best_run.data.tags.get('mlflow.runName')}  RMSE: {best_run.data.metrics['rmse']:.4f}")
```
**What it does:** Prints a summary of the best model.

**Breaking down the f-string:**
1. **`best_run.data.tags.get('mlflow.runName')`**
   - Navigate to the `tags` dictionary
   - Get the value of the `'mlflow.runName'` key
   - Example: `"auto_arima_best"`

2. **`best_run.data.metrics['rmse']`**
   - Navigate to the `metrics` dictionary
   - Get the RMSE value
   - Example: `4.371798`

3. **`:.4f`**
   - Format the RMSE to 4 decimal places
   - `4.371798` → `4.3718`

**Example output:**
```
Best run: auto_arima_best  RMSE: 4.3718
```

---

## 🎯 Summary: What This Cell Does

1. **Connects to MLflow** via `MlflowClient()`
2. **Finds your experiment** by name (`weather-sarima`)
3. **Searches all runs** in that experiment
4. **Sorts by RMSE** (lowest = best)
5. **Selects the top model** (runs[0])
6. **Extracts the run ID** for model loading
7. **Prints confirmation** showing which model won

---

## 💡 Key Concepts

**MLflow Run:** A single execution of model training code with specific hyperparameters
- Contains: parameters, metrics, artifacts, tags
- Has unique run_id for traceability

**MLflow Client:** Python API to query MLflow tracking data
- Search runs, compare metrics, download models
- Alternative to using the MLflow UI manually

**RMSE-based Selection:** Automatic model selection
- No manual comparison needed
- Programmatically choose best model
- Can be integrated into CI/CD pipelines

---

## 🔗 What Happens Next?

In **Cell 4**, this `best_run_id` is used to:
1. Download the trained model file (`sarima_model.pkl`)
2. Load it into memory
3. Generate 30-day forecasts
4. Save predictions to `weather_forecast_gold` table

This creates **full lineage tracking**: every forecast row links back to the exact model run that produced it!

In [0]:
# Cell 2: Note - model was logged as artifact, not MLflow model
# The sarima_model.pkl was logged via log_artifact(), not log_model()
# so it cannot be registered to the model registry.
# Cell 3 downloads it directly as an artifact for inference.
print(f"Best model artifact available at: runs:/{best_run_id}/sarima_model.pkl")
print("Model will be loaded directly from artifacts for inference.")




## 📦 What Was Happening in the Below Cell?

The cell prints information about the best SARIMA model artifact found in MLflow:

- It shows the path to the model file (`sarima_model.pkl`) stored as an artifact for the best run.
- Explains that the model was logged as a plain artifact (not as an MLflow model), so it can't be registered in the MLflow Model Registry.
- Indicates that the model will be loaded directly from the artifact for inference in the next step.

This sets up the workflow for loading the model and generating forecasts in the following cell.

In [0]:
# Cell 3: Batch inference — load model and forecast next 30 days
artifact_path = client.download_artifacts(best_run_id, "sarima_model.pkl", "/tmp/")
with open("/tmp/sarima_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

forecast_steps = 30
forecast = loaded_model.forecast(steps=forecast_steps)
forecast_conf = loaded_model.get_forecast(steps=forecast_steps).conf_int()

last_date = pd.to_datetime("2023-12-31")
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=forecast_steps)

forecast_pdf = pd.DataFrame({
    "date": future_dates,
    "forecast_temp": forecast.values,
    "lower_ci": forecast_conf.iloc[:, 0].values,
    "upper_ci": forecast_conf.iloc[:, 1].values,
    "model_run_id": best_run_id
})

# Write to Gold Delta table
df_forecast = spark.createDataFrame(forecast_pdf)
df_forecast.write.format("delta").mode("overwrite").saveAsTable("weather_forecast_gold")
display(df_forecast)